#  Ablation: Effect of Batch Size on Contrastive Learning

**Research question:**  
> Does a larger batch size improve retrieval performance in contrastive learning?

**Why batch size matters for CLIP:**  
The InfoNCE loss treats all other items in the batch as negatives.  
A batch of 128 gives each query 127 negatives to distinguish from.  
A batch of 32 gives only 31 negatives — an easier task, potentially weaker learning.





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/prajwl7676/mini-clip-DLAI.git

%cd /content/mini-clip-DLAI
!pip install -q -r requirements.txt

import sys, os
sys.path.insert(0, '/content/mini-clip-DLAI')

In [ ]:
%cd /content/mini-clip-DLAI

In [ ]:

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import DistilBertTokenizer

from src.model_architecture    import CLIPModel
from src.dataset  import FlickrDataset, get_transform, build_loaders
from src.train    import train_epoch, validate, save_checkpoint, load_checkpoint
from src.evaluate import encode_dataset, recall_at_k, print_results

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
DRIVE_DIR = '/content/drive/MyDrive/mini-clip-dlai/ablation'
os.makedirs(DRIVE_DIR, exist_ok=True)

---
##  Shared configuration



In [ ]:
# Fixed across all runs
SHARED_CFG = {
    'total_samples': 10_000,
    'n_train':        8_000,
    'n_val':          1_000,
    'embedding_dim':    256,
    'num_epochs':         8,
    'lr':             1e-4,
    'lr_head':        1e-3,
    'weight_decay':   1e-2,
    'max_grad_norm':   1.0,
    'num_workers':      2,
}

# The variable we are ablating
BATCH_SIZES = [32, 64, 128]

print('Shared config:')
for k, v in SHARED_CFG.items():
    print(f'  {k:<20}: {v}')
print(f'\nBatch sizes to test: {BATCH_SIZES}')

---
##  Load data

In [ ]:
full_ds   = load_dataset('nlphuji/flickr30k', split='test')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

n_train = SHARED_CFG['n_train']
n_val   = SHARED_CFG['n_val']

train_hf = full_ds.select(range(0, n_train))
val_hf   = full_ds.select(range(n_train, n_train + n_val))
test_hf  = full_ds.select(range(n_train + n_val, SHARED_CFG['total_samples']))

train_ds = FlickrDataset(train_hf, tokenizer, get_transform(train=True))
val_ds   = FlickrDataset(val_hf,   tokenizer, get_transform(train=False))
test_ds  = FlickrDataset(test_hf,  tokenizer, get_transform(train=False))

print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}  |  Test: {len(test_ds):,}')

---
##  Training function for one ablation run


In [ ]:
import math
def run_ablation(batch_size: int, cfg: dict) -> dict:
    """
    Train a fresh CLIPModel with the given batch_size.
    Returns the final val loss and test Recall@K results.
    """
    print(f'\n{"+" * 60}')
    print(f'  ABLATION RUN — batch_size={batch_size}')
    print(f'  Random loss baseline: log({batch_size}) = {math.log(batch_size):.3f}')
    print(f'{"+" * 60}')

    # ── DataLoaders ──────────────────────────────────────
    train_loader, val_loader, test_loader = build_loaders(
        train_ds, val_ds, test_ds,
        batch_size  = batch_size,
        num_workers = cfg['num_workers'],
    )

    # ── Fresh model ───────────────────────────────────────
    # IMPORTANT: fresh model each run — same random seed for fairness
    torch.manual_seed(42)
    model = CLIPModel(embedding_dim=cfg['embedding_dim']).to(device)

    # ── Optimizer + scheduler ─────────────────────────────
    projection_params = (
        list(model.image_encoder.projection.parameters()) +
        list(model.text_encoder.projection.parameters())  +
        [model.log_temperature]
    )
    encoder_params = [
        p for p in model.parameters()
        if p.requires_grad and not any(p is pp for pp in projection_params)
    ]
    optimizer = torch.optim.AdamW([
        {'params': encoder_params,    'lr': cfg['lr'],      'weight_decay': cfg['weight_decay']},
        {'params': projection_params, 'lr': cfg['lr_head'], 'weight_decay': 0.0},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['num_epochs'], eta_min=1e-6
    )

    # ── Training loop ─────────────────────────────────────
    loss_history  = {'train': [], 'val': []}
    best_val_loss = float('inf')
    best_ckpt     = os.path.join(DRIVE_DIR, f'best_bs{batch_size}.pt')

    for epoch in range(cfg['num_epochs']):
        tr = train_epoch(model, train_loader, optimizer, device, cfg['max_grad_norm'])
        vl = validate(model, val_loader, device)
        scheduler.step()

        loss_history['train'].append(tr['loss'])
        loss_history['val'].append(vl['loss'])

        improved = ' ← best' if vl['loss'] < best_val_loss else ''
        print(f'  Epoch {epoch+1:02d}/{cfg["num_epochs"]:02d} | '
              f'train {tr["loss"]:.4f} | val {vl["loss"]:.4f}{improved}')

        if vl['loss'] < best_val_loss:
            best_val_loss = vl['loss']
            save_checkpoint(model, optimizer, epoch,
                            tr['loss'], vl['loss'], loss_history,
                            save_path=best_ckpt)

    # ── Load best and evaluate ────────────────────────────
    ckpt = load_checkpoint(model, optimizer, best_ckpt, device)
    embeddings = encode_dataset(model, test_loader, device)
    results    = recall_at_k(
        embeddings['image_embeddings'],
        embeddings['text_embeddings'],
        ks=[1, 5, 10]
    )

    print_results(results, title=f'  batch_size={batch_size}')

    return {
        'batch_size':   batch_size,
        'best_val_loss': best_val_loss,
        'results':      results,
        'loss_history': loss_history,
    }


print('run_ablation() defined.')

---
##  three ablation experiments



In [ ]:
ablation_results = []

for bs in BATCH_SIZES:
    run_result = run_ablation(bs, SHARED_CFG)
    ablation_results.append(run_result)

print('\nAll ablation runs complete.')

---
##  Ablation results table

In [ ]:
print('\n=== Ablation Results: Effect of Batch Size ===')
print(f'{"Batch size":<12} {"t2i R@1":>9} {"t2i R@5":>9} {"t2i R@10":>10} {"i2t R@1":>9} {"Val Loss":>10}')
print('─' * 62)

for r in ablation_results:
    bs  = r['batch_size']
    res = r['results']
    vl  = r['best_val_loss']
    print(
        f'{bs:<12} '
        f'{res["t2i_R@1"]:>8.2f}% '
        f'{res["t2i_R@5"]:>8.2f}% '
        f'{res["t2i_R@10"]:>9.2f}% '
        f'{res["i2t_R@1"]:>8.2f}% '
        f'{vl:>10.4f}'
    )
print('─' * 62)

---
##  Plot ablation results

In [ ]:
batch_sizes  = [r['batch_size'] for r in ablation_results]
t2i_r1       = [r['results']['t2i_R@1'] for r in ablation_results]
t2i_r5       = [r['results']['t2i_R@5'] for r in ablation_results]
i2t_r1       = [r['results']['i2t_R@1'] for r in ablation_results]
val_losses   = [r['best_val_loss'] for r in ablation_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Recall@K vs batch size
ax = axes[0]
ax.plot(batch_sizes, t2i_r1, 'o-', color='steelblue',   lw=2, ms=8, label='t2i R@1')
ax.plot(batch_sizes, t2i_r5, 's-', color='mediumseagreen', lw=2, ms=8, label='t2i R@5')
ax.plot(batch_sizes, i2t_r1, '^-', color='tomato',      lw=2, ms=8, label='i2t R@1')
ax.set_xlabel('Batch size')
ax.set_ylabel('Recall (%)')
ax.set_title('Retrieval Performance vs Batch Size')
ax.set_xticks(batch_sizes)
ax.legend()
ax.grid(True, alpha=0.3)

# Right: Val loss vs batch size
ax2 = axes[1]
bars = ax2.bar([str(bs) for bs in batch_sizes], val_losses,
               color=['#B0BEC5', '#1D9E75', '#7F77DD'], edgecolor='white', width=0.5)
ax2.set_xlabel('Batch size')
ax2.set_ylabel('Best Validation Loss')
ax2.set_title('Validation Loss vs Batch Size')
ax2.grid(True, axis='y', alpha=0.3)

for bar, vl in zip(bars, val_losses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{vl:.4f}', ha='center', va='bottom', fontsize=9)

# Add log(B) baseline markers
for bs in batch_sizes:
    ax2.axhline(math.log(bs), linestyle=':', color='grey', alpha=0.4)

plt.suptitle('Ablation Study: Effect of Batch Size on Mini-CLIP', fontsize=12)
plt.tight_layout()
plt.savefig('assets/ablation_batch_size.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to assets/ablation_batch_size.png')

---
##  Loss curves for all three runs

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
colors = ['#B0BEC5', '#1D9E75', '#7F77DD']

for ax, r, color in zip(axes, ablation_results, colors):
    bs      = r['batch_size']
    history = r['loss_history']
    epochs  = range(1, len(history['train']) + 1)

    ax.plot(epochs, history['train'], 'o-', color=color,  lw=2, ms=4, label='train')
    ax.plot(epochs, history['val'],   's--', color=color, lw=2, ms=4, alpha=0.6, label='val')
    ax.axhline(math.log(bs), color='grey', linestyle=':', lw=1,
               label=f'log({bs})={math.log(bs):.2f}')
    ax.set_title(f'batch_size = {bs}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('InfoNCE Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training curves by batch size', fontsize=12)
plt.tight_layout()
plt.savefig('assets/ablation_loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()